# 06. Evaluation and visualization

**Goal:** Dice scoring per class with `nanmean`, absent-class handling, and prediction-versus-ground-truth overlays.

In [ ]:
# bootstrap: make course_utils + scripts importable from any working directory,
# use the inline backend so figures render, and regenerate the phantom if missing.
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO = _find_repo_root(Path.cwd())
for _p in (str(REPO), str(REPO / "scripts")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

DATA = REPO / "assets" / "data" / "Dataset999_Phantom"
PRE = REPO / "assets" / "precomputed"

if not (DATA / "imagesTr" / "PHANTOM_001_0000.nii.gz").exists():
    import generate_phantom
    generate_phantom.generate(REPO / "assets" / "data")

print("repo root:", REPO.name)


## Dice = 2 |A and B| / (|A| + |B|)

A hand-built half-overlap case scores exactly 0.5.

In [ ]:
from course_utils.dsc import dice_for_label, dice_per_label, nanmean_dice
a = np.array([1, 1, 0, 0]); b = np.array([1, 0, 1, 0])
print('half overlap :', dice_for_label(a, b, 1))
print('perfect      :', dice_for_label(a, a, 1))
print('absent class :', dice_for_label(np.zeros(4, int), np.zeros(4, int), 1))

## Absent classes and `nanmean`

If a class is in neither the prediction nor the reference, Dice is undefined (NaN). `nanmean` drops it. Ignoring absent classes is a decision that changes the reported number, so state it.

In [ ]:
scores = dice_per_label(a, b, labels=[1, 2])   # label 2 is absent
print('per label:', scores)
print('nanmean over {1, 2}:', nanmean_dice(scores, labels=[1, 2]), '(equals the label-1 score)')

## Score the illustrative prediction on the phantom

In [ ]:
import json
import nibabel as nib
ct = np.asarray(nib.load(str(DATA / 'imagesTr' / 'PHANTOM_001_0000.nii.gz')).dataobj, dtype=np.float32)
gt = np.rint(np.asarray(nib.load(str(DATA / 'labelsTr' / 'PHANTOM_001.nii.gz')).dataobj)).astype(int)
from teaching_fixtures import illustrative_prediction
pred = illustrative_prediction(gt)
scores = dice_per_label(pred, gt, labels=[1])
summary = {'metric_per_label': {str(k): v for k, v in scores.items()},
           'foreground_mean_dice': nanmean_dice(scores, labels=[1])}
print(json.dumps(summary, indent=2))   # same shape as nnU-Net's summary.json

In [ ]:
from course_utils.viz import overlay_mask_on_slice
z = gt.shape[2] // 2
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
overlay_mask_on_slice(ct, gt, z=z, ax=axes[0]); axes[0].set_title('ground truth')
overlay_mask_on_slice(ct, pred, z=z, ax=axes[1]); axes[1].set_title('prediction')
fig.tight_layout()
plt.show()

## Recap
1. Score Dice per label and aggregate with `nanmean`, dropping absent-class NaNs (a stated policy).
2. Look at overlays, not just the number. nnU-Net writes these metrics to `summary.json`.